# Random Forest Classification for TLS Profiling

This notebook evaluates a supervised Random Forest baseline on extracted TLS traffic features. The dataset paths are parameterized so the same experiment can be run on one or more parquet partitions. Unlike the anomaly-detection baselines, this model is trained to classify each flow directly into one of four categories: `system`, `malware`, `application`, or `unknown`.


In [1]:
import sys
from pathlib import Path
sys.path.append('../../src')


### PARAMETERS:


In [2]:
train_features_path = ["../data-ext/malware_train.parquet", "../data-ext/apps_train.parquet"]
val_features_path = ["../data-ext/malware_val.parquet", "../data-ext/apps_val.parquet"]
test_features_path = ["../data-ext/malware_test.parquet", "../data-ext/apps_test.parquet"]
train_labels_path = ["../data-ext/malware_train_labels.parquet", "../data-ext/apps_train_labels.parquet"]
val_labels_path = ["../data-ext/malware_val_labels.parquet", "../data-ext/apps_val_labels.parquet"]
test_labels_path = ["../data-ext/malware_test_labels.parquet", "../data-ext/apps_test_labels.parquet"]


## Feature Groups

- `FLOW`: basic bidirectional flow statistics such as bytes, packets, rates, and duration (`bs`, `ps`, `br`, `pr`, `td`)
- `CTLS`: client-side TLS metadata (`tls.cver`, `tls.ccs`, `tls.cext`, `tls.csg`, `tls.alpn`, `tls.csv`)
- `STLS`: server-side TLS metadata (`tls.sver`, `tls.scs`, `tls.sext`, `tls.ssv`)
- `REC`: ordered sequence of signed TLS record lengths (`tls.rec`, first 20 records)

The experiments below sweep over every non-empty combination of these groups.


In [3]:
feature_groups = {
    "FLOW": ["bs", "ps", "br", "pr", "td"],
    "CTLS": ["tls.cver", "tls.ccs", "tls.cext", "tls.csg", "tls.alpn", "tls.csv"],
    "STLS": ["tls.sver", "tls.scs", "tls.sext", "tls.ssv"],
    "REC": ["tls.rec"],
}

FLOW = feature_groups["FLOW"]
CTLS = feature_groups["CTLS"]
STLS = feature_groups["STLS"]
REC = feature_groups["REC"]


In [4]:
import pandas as pd
from pathlib import Path
from tls_profiling.preprocessing import build_and_fit_preprocessor

def read_parquet_files(paths):
    if isinstance(paths, (str, Path)):
        paths = [paths]
    return pd.concat([pd.read_parquet(Path(path)) for path in paths], ignore_index=True)

print("Loading extracted features from parameterized parquet paths.")
df_train = read_parquet_files(train_features_path)
df_val = read_parquet_files(val_features_path)
df_test = read_parquet_files(test_features_path)
df_train_label = read_parquet_files(train_labels_path)
df_val_label = read_parquet_files(val_labels_path)
df_test_label = read_parquet_files(test_labels_path)

preprocessor = build_and_fit_preprocessor(df_train)
print("Built preprocessor from df_train.")


Loading extracted features from parameterized parquet paths.
Built preprocessor from df_train.


In [5]:
from tls_profiling.baselines.model_random_forest import RandomForestBaseline, Config
import numpy as np

from sklearn.metrics import average_precision_score, f1_score, roc_auc_score
from sklearn.preprocessing import label_binarize

CLASS_NAMES = ["system", "malware", "application", "unknown"]
CLASS_TO_INDEX = {label: idx for idx, label in enumerate(CLASS_NAMES)}
INDEX_TO_CLASS = {idx: label for label, idx in CLASS_TO_INDEX.items()}

RF_CONFIG_CANDIDATES = {
    "balanced_deep": Config(
        n_estimators=200,
        max_depth=None,
        min_samples_leaf=1,
        class_weight="balanced_subsample",
        max_train_samples=100_000,
    ),
    "balanced_shallow": Config(
        n_estimators=200,
        max_depth=30,
        min_samples_leaf=2,
        class_weight="balanced_subsample",
        max_train_samples=100_000,
    ),
}

def encode_labels(series: pd.Series) -> np.ndarray:
    return series.map(CLASS_TO_INDEX).to_numpy(dtype=np.int64)

def decode_labels(values: np.ndarray) -> list[str]:
    return [INDEX_TO_CLASS[int(value)] for value in values]

def select_feature_set(x, feature_set):
    if not feature_set:
        return x

    selected_columns = [
        column for column in x.columns
        if any(column.startswith(prefix) for prefix in feature_set)
    ]
    return x.loc[:, selected_columns]

def compute_multiclass_scores(y_true, y_pred, y_score):
    y_true_bin = label_binarize(y_true, classes=np.arange(len(CLASS_NAMES)))
    per_class_f1 = f1_score(
        y_true,
        y_pred,
        average=None,
        labels=np.arange(len(CLASS_NAMES)),
    )

    return {
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "macro_auc": roc_auc_score(
            y_true,
            y_score,
            labels=np.arange(len(CLASS_NAMES)),
            multi_class="ovr",
            average="macro",
        ),
        "macro_ap": average_precision_score(y_true_bin, y_score, average="macro"),
        "system_f1": per_class_f1[CLASS_TO_INDEX["system"]],
        "malware_f1": per_class_f1[CLASS_TO_INDEX["malware"]],
        "application_f1": per_class_f1[CLASS_TO_INDEX["application"]],
        "unknown_f1": per_class_f1[CLASS_TO_INDEX["unknown"]],
    }

def choose_best_model(x_train, y_train, x_val, y_val):
    best_name = None
    best_model = None
    best_val_macro_f1 = -np.inf

    for config_name, config in RF_CONFIG_CANDIDATES.items():
        model = RandomForestBaseline(config)
        model.fit(x_train, y_train)

        val_pred = model.predict(x_val)
        val_macro_f1 = f1_score(y_val, val_pred, average="macro")

        if val_macro_f1 > best_val_macro_f1:
            best_name = config_name
            best_model = model
            best_val_macro_f1 = val_macro_f1

    return best_name, best_model, float(best_val_macro_f1)

def evaluate(feature_set):
    y_train = encode_labels(df_train_label["category"])
    y_val = encode_labels(df_val_label["category"])
    y_test = encode_labels(df_test_label["category"])

    x_train_transformed = select_feature_set(preprocessor.transform(df_train), feature_set)
    x_val_transformed = select_feature_set(preprocessor.transform(df_val), feature_set)
    x_test_transformed = select_feature_set(preprocessor.transform(df_test), feature_set)

    best_config_name, model, val_macro_f1 = choose_best_model(
        x_train_transformed,
        y_train,
        x_val_transformed,
        y_val,
    )

    y_pred = model.predict(x_test_transformed)
    y_score = model.predict_proba(x_test_transformed)

    return y_test, y_pred, y_score, val_macro_f1, best_config_name

def debug_csv(feature_set, y_test, y_pred, y_score):
    feature_set_name = "_".join(feature_set)
    output_path = f"tmp/rf_{feature_set_name}.csv"

    pd.DataFrame({
        "y_test": decode_labels(y_test),
        "y_pred": decode_labels(y_pred),
        "p_system": y_score[:, CLASS_TO_INDEX["system"]],
        "p_malware": y_score[:, CLASS_TO_INDEX["malware"]],
        "p_application": y_score[:, CLASS_TO_INDEX["application"]],
        "p_unknown": y_score[:, CLASS_TO_INDEX["unknown"]],
    }).to_csv(output_path, index=False)

def compute_scores(feature_set):
    y_test, y_pred, y_score, val_macro_f1, best_config_name = evaluate(feature_set)
    scores = compute_multiclass_scores(y_test, y_pred, y_score)

    debug_csv(feature_set, y_test, y_pred, y_score)
    return {
        "best_config": best_config_name,
        "val_macro_f1": val_macro_f1,
        **scores,
    }


## Evaluation

This baseline uses the same train, validation, and test splits as the anomaly-detection notebooks, but the task is now multiclass classification rather than one-class profiling.

Protocol:

- `train`: fit the Random Forest on labeled traffic from all target categories
- `val`: select the best hyperparameter configuration using macro F1
- `test`: report multiclass metrics for the selected configuration on held-out traffic

The target classes in this notebook are `system`, `malware`, `application`, and `unknown`.


In [6]:
from itertools import combinations

rows = []

group_names = list(feature_groups)
for size in range(1, len(group_names) + 1):
    for group_combo in combinations(group_names, size):
        selected_features = []
        for group_name in group_combo:
            selected_features.extend(feature_groups[group_name])

        feature_set_name = ", ".join(group_combo)
        scores = compute_scores(selected_features)

        rows.append({
            "FeatureSet": feature_set_name,
            "BestConfig": scores["best_config"],
            "ValMacroF1": scores["val_macro_f1"],
            "MacroF1": scores["macro_f1"],
            "MacroAucScore": scores["macro_auc"],
            "MacroApScore": scores["macro_ap"],
            "SystemF1": scores["system_f1"],
            "MalwareF1": scores["malware_f1"],
            "ApplicationF1": scores["application_f1"],
            "UnknownF1": scores["unknown_f1"],
        })

results_df = pd.DataFrame(
    rows,
    columns=[
        "FeatureSet",
        "BestConfig",
        "ValMacroF1",
        "MacroF1",
        "MacroAucScore",
        "MacroApScore",
        "SystemF1",
        "MalwareF1",
        "ApplicationF1",
        "UnknownF1",
    ],
)
display(results_df)


,FeatureSet,BestConfig,ValMacroF1,MacroF1,MacroAucScore,MacroApScore,SystemF1,MalwareF1,ApplicationF1,UnknownF1
0,FLOW,balanced_deep,0.751721,0.712146,0.953190,0.788712,0.926034,0.848088,0.471938,0.602525
1,CTLS,balanced_deep,0.503844,0.479295,0.822524,0.520449,0.868052,0.825059,0.095178,0.128892
2,STLS,balanced_deep,0.531908,0.486781,0.844214,0.573240,0.813886,0.650944,0.065107,0.417189
3,REC,balanced_deep,0.779104,0.684448,0.961198,0.808788,0.919817,0.829762,0.318963,0.669250
4,"FLOW, CTLS",balanced_deep,0.788290,0.761931,0.961284,0.823676,0.927916,0.891340,0.539792,0.688675
5,"FLOW, STLS",balanced_deep,0.778159,0.711416,0.957365,0.789810,0.927680,0.864276,0.404365,0.649344
6,"FLOW, REC",balanced_deep,0.801396,0.700172,0.965239,0.829742,0.925015,0.847737,0.342348,0.685587
7,"CTLS, STLS",balanced_deep,0.589552,0.571065,0.885606,0.638934,0.817593,0.861079,0.152425,0.453161
8,"CTLS, REC",balanced_deep,0.810138,0.788427,0.961595,0.844470,0.949877,0.917325,0.580278,0.706229
9,"STLS, REC",balanced_deep,0.796089,0.699737,0.962445,0.825545,0.922160,0.842944,0.351336,0.682508


## Overall Results

The table below ranks feature sets by `MacroF1`, which is the primary metric for this multiclass baseline. `MacroAucScore` and `MacroApScore` are included to make the supervised baseline easier to compare at a high level with the anomaly-detection notebooks.


In [7]:
overall_df = results_df.sort_values("MacroF1", ascending=False)
display(overall_df)

,FeatureSet,BestConfig,ValMacroF1,MacroF1,MacroAucScore,MacroApScore,SystemF1,MalwareF1,ApplicationF1,UnknownF1
14,"FLOW, CTLS, STLS, REC",balanced_deep,0.820877,0.821413,0.969045,0.869900,0.956472,0.921546,0.687230,0.720405
13,"CTLS, STLS, REC",balanced_deep,0.812490,0.805574,0.965412,0.852050,0.952354,0.911026,0.659280,0.699636
11,"FLOW, CTLS, REC",balanced_deep,0.819869,0.804270,0.968695,0.863043,0.953899,0.923947,0.623180,0.716054
8,"CTLS, REC",balanced_deep,0.810138,0.788427,0.961595,0.844470,0.949877,0.917325,0.580278,0.706229
10,"FLOW, CTLS, STLS",balanced_deep,0.797762,0.776920,0.964219,0.837819,0.929190,0.893083,0.583402,0.702005
4,"FLOW, CTLS",balanced_deep,0.788290,0.761931,0.961284,0.823676,0.927916,0.891340,0.539792,0.688675
12,"FLOW, STLS, REC",balanced_deep,0.809379,0.725895,0.966568,0.844074,0.919546,0.854101,0.442829,0.687103
0,FLOW,balanced_deep,0.751721,0.712146,0.953190,0.788712,0.926034,0.848088,0.471938,0.602525
5,"FLOW, STLS",balanced_deep,0.778159,0.711416,0.957365,0.789810,0.927680,0.864276,0.404365,0.649344
6,"FLOW, REC",balanced_deep,0.801396,0.700172,0.965239,0.829742,0.925015,0.847737,0.342348,0.685587


## System Category

This table isolates how well each feature set classifies the `system` category. A high `SystemF1` means the supervised classifier can separate system traffic cleanly from both malware and application traffic.


In [8]:
system_df = results_df.sort_values("SystemF1", ascending=False)
display(system_df)

,FeatureSet,BestConfig,ValMacroF1,MacroF1,MacroAucScore,MacroApScore,SystemF1,MalwareF1,ApplicationF1,UnknownF1
14,"FLOW, CTLS, STLS, REC",balanced_deep,0.820877,0.821413,0.969045,0.869900,0.956472,0.921546,0.687230,0.720405
11,"FLOW, CTLS, REC",balanced_deep,0.819869,0.804270,0.968695,0.863043,0.953899,0.923947,0.623180,0.716054
13,"CTLS, STLS, REC",balanced_deep,0.812490,0.805574,0.965412,0.852050,0.952354,0.911026,0.659280,0.699636
8,"CTLS, REC",balanced_deep,0.810138,0.788427,0.961595,0.844470,0.949877,0.917325,0.580278,0.706229
10,"FLOW, CTLS, STLS",balanced_deep,0.797762,0.776920,0.964219,0.837819,0.929190,0.893083,0.583402,0.702005
4,"FLOW, CTLS",balanced_deep,0.788290,0.761931,0.961284,0.823676,0.927916,0.891340,0.539792,0.688675
5,"FLOW, STLS",balanced_deep,0.778159,0.711416,0.957365,0.789810,0.927680,0.864276,0.404365,0.649344
0,FLOW,balanced_deep,0.751721,0.712146,0.953190,0.788712,0.926034,0.848088,0.471938,0.602525
6,"FLOW, REC",balanced_deep,0.801396,0.700172,0.965239,0.829742,0.925015,0.847737,0.342348,0.685587
9,"STLS, REC",balanced_deep,0.796089,0.699737,0.962445,0.825545,0.922160,0.842944,0.351336,0.682508


## Malware Category

This section focuses on the `malware` class. Strong `MalwareF1` values indicate that the selected feature family helps the Random Forest carve out malware traffic as a distinct supervised class rather than only separating it from one chosen normal profile.


In [9]:
malware_df = results_df.sort_values("MalwareF1", ascending=False)
display(malware_df)

,FeatureSet,BestConfig,ValMacroF1,MacroF1,MacroAucScore,MacroApScore,SystemF1,MalwareF1,ApplicationF1,UnknownF1
11,"FLOW, CTLS, REC",balanced_deep,0.819869,0.804270,0.968695,0.863043,0.953899,0.923947,0.623180,0.716054
14,"FLOW, CTLS, STLS, REC",balanced_deep,0.820877,0.821413,0.969045,0.869900,0.956472,0.921546,0.687230,0.720405
8,"CTLS, REC",balanced_deep,0.810138,0.788427,0.961595,0.844470,0.949877,0.917325,0.580278,0.706229
13,"CTLS, STLS, REC",balanced_deep,0.812490,0.805574,0.965412,0.852050,0.952354,0.911026,0.659280,0.699636
10,"FLOW, CTLS, STLS",balanced_deep,0.797762,0.776920,0.964219,0.837819,0.929190,0.893083,0.583402,0.702005
4,"FLOW, CTLS",balanced_deep,0.788290,0.761931,0.961284,0.823676,0.927916,0.891340,0.539792,0.688675
5,"FLOW, STLS",balanced_deep,0.778159,0.711416,0.957365,0.789810,0.927680,0.864276,0.404365,0.649344
7,"CTLS, STLS",balanced_deep,0.589552,0.571065,0.885606,0.638934,0.817593,0.861079,0.152425,0.453161
12,"FLOW, STLS, REC",balanced_deep,0.809379,0.725895,0.966568,0.844074,0.919546,0.854101,0.442829,0.687103
0,FLOW,balanced_deep,0.751721,0.712146,0.953190,0.788712,0.926034,0.848088,0.471938,0.602525


## Application Category

This section focuses on the `application` class. Strong `ApplicationF1` values indicate that the selected feature family helps the Random Forest separate application traffic from both system and malware flows in the combined dataset.


In [10]:
application_df = results_df.sort_values("ApplicationF1", ascending=False)
display(application_df)


,FeatureSet,BestConfig,ValMacroF1,MacroF1,MacroAucScore,MacroApScore,SystemF1,MalwareF1,ApplicationF1,UnknownF1
14,"FLOW, CTLS, STLS, REC",balanced_deep,0.820877,0.821413,0.969045,0.869900,0.956472,0.921546,0.687230,0.720405
13,"CTLS, STLS, REC",balanced_deep,0.812490,0.805574,0.965412,0.852050,0.952354,0.911026,0.659280,0.699636
11,"FLOW, CTLS, REC",balanced_deep,0.819869,0.804270,0.968695,0.863043,0.953899,0.923947,0.623180,0.716054
10,"FLOW, CTLS, STLS",balanced_deep,0.797762,0.776920,0.964219,0.837819,0.929190,0.893083,0.583402,0.702005
8,"CTLS, REC",balanced_deep,0.810138,0.788427,0.961595,0.844470,0.949877,0.917325,0.580278,0.706229
4,"FLOW, CTLS",balanced_deep,0.788290,0.761931,0.961284,0.823676,0.927916,0.891340,0.539792,0.688675
0,FLOW,balanced_deep,0.751721,0.712146,0.953190,0.788712,0.926034,0.848088,0.471938,0.602525
12,"FLOW, STLS, REC",balanced_deep,0.809379,0.725895,0.966568,0.844074,0.919546,0.854101,0.442829,0.687103
5,"FLOW, STLS",balanced_deep,0.778159,0.711416,0.957365,0.789810,0.927680,0.864276,0.404365,0.649344
9,"STLS, REC",balanced_deep,0.796089,0.699737,0.962445,0.825545,0.922160,0.842944,0.351336,0.682508
